# mmraz-qwen3-probe-artifact-steering-question-options-answer-vast

Direct-run Vast notebook for the Qwen steering experiment.

What it does:
- loads the latest `qwen3_4b_question_options_answer_probe_artifacts_*.npz` bundle by default,
- defaults to `explicit_train_only` + `mean_answer_tokens` + `mm_steering_vectors`,
- evaluates baseline plus steering on the expanded `explicit_test` and `implicit_full` datasets,
- uses the training-style prompt format:
  - question line,
  - `Options:` with stripped option texts,
  - `Answer:` on its own line,
- sweeps layers `10, 14, 18, 22, 26` and signed strengths `+-2, +-4, +-8, +-16, +-32`,
- keeps `do_sample=False`, and
- saves full logs, summaries, config, per-layer probe-slice stats, partial checkpoints, and heatmaps.

Important default:
- the loaded probe artifact must use the new pair-level + chat-template + no-thinking-trace format (`artifact_format_version >= 4`),
- the steering run reuses the exact held-out explicit pair indices saved in that artifact, and
- older example-level or non-chat-template probe bundles are rejected on load.


In [1]:
from pathlib import Path
import runpy

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from current working directory.')


ROOT = find_repo_root(Path.cwd())
script_path = ROOT / 'scripts' / 'vast' / 'run_mmraz_qwen3_probe_artifact_steering.py'
script_ns = runpy.run_path(str(script_path))
run_experiment = script_ns['run_experiment']

print('Repo root   :', ROOT)
print('Helper script:', script_path)


/opt/conda/envs/my_remote_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root   : /workspace
Helper script: /workspace/scripts/vast/run_mmraz_qwen3_probe_artifact_steering.py


In [2]:
CONFIG_OVERRIDES = {
    'model_name': 'Qwen/Qwen3-4B',
    'train_regime': 'explicit_train_only',
    'feature_name': 'mean_answer_tokens',
    'vector_key': 'mm_steering_vectors',
    'layers_to_test': [10, 14, 18, 22, 26],
    'use_chat_template': True,
    'disable_thinking_trace': True,
    'strengths': [2.0, 4.0, 8.0, 16.0, 32.0],
    'pair_split_random_state': 42,
    'patch_prompt_last_only': True,
    'patch_generation_tokens': True,
    'max_new_tokens': 32,
    'quick_mode': False,
    'artifact_path': None,
    'metadata_path': None,
    'reuse_existing_results': True,
    'reuse_result_search_roots': None,
}

display(pd.DataFrame([
    {'key': key, 'value': value if not isinstance(value, list) else ', '.join(str(v) for v in value)}
    for key, value in CONFIG_OVERRIDES.items()
]))


,key,value
0,model_name,Qwen/Qwen3-4B
1,train_regime,explicit_train_only
2,feature_name,mean_answer_tokens
3,vector_key,mm_steering_vectors
4,layers_to_test,"10, 14, 18, 22, 26"
5,use_chat_template,True
6,disable_thinking_trace,True
7,strengths,"2.0, 4.0, 8.0, 16.0, 32.0"
8,pair_split_random_state,42
9,patch_prompt_last_only,True


In [3]:
result = run_experiment(CONFIG_OVERRIDES)


Repo root: /workspace
Device: cuda
Model: Qwen/Qwen3-4B
Expanded explicit dataset: /workspace/data/raw/temporal_scope_AB_randomized/temporal_scope_explicit_expanded_500.json
Expanded implicit dataset: /workspace/data/raw/temporal_scope_AB_randomized/temporal_scope_implicit_expanded_300.json
Probe artifact: /workspace/results/qwen_question_options_answer_probe_variations_vast/20260405-185421/qwen3_4b_question_options_answer_probe_artifacts_20260405-185421.npz
Probe metadata: /workspace/results/qwen_question_options_answer_probe_variations_vast/20260405-185421/qwen3_4b_question_options_answer_probe_metadata_20260405-185421.json
Dataset source: expanded
Use chat template: True
Disable thinking trace: True
Quick mode: False
Artifact split metadata: pair_level_80_20 | version = 4
Explicit train pairs from artifact: 400
Explicit test pairs from artifact : 100 | eval subset = 100
Implicit pairs: 300 | eval subset = 300
Reuse existing results: True
Existing result search roots: ['/workspace/re

'The read operation timed out' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen3-4B/resolve/main/vocab.json
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen3-4B/resolve/main/tokenizer.json
Retrying in 1s [Retry 1/5].
Loading weights: 100%|██████████| 398/398 [00:03<00:00, 124.26it/s]


Loaded model | n_layers = 36 | hidden_size = 2560
A token IDs: [32, 362]
B token IDs: [33, 425]


baseline generations: 100%|██████████| 400/400 [00:00<00:00, 34129.17gen/s]


[explicit_test] reused baseline from /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_colab/20260405-194733/mmraz_qwen3_probe_artifact_steering_question_options_answer_logs_20260405-194733.csv
[implicit_full] reused baseline from /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_colab/20260405-194733/mmraz_qwen3_probe_artifact_steering_question_options_answer_logs_20260405-194733.csv
Checkpoint saved (after baseline): /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_vast/20260406-121506/mmraz_qwen3_probe_artifact_steering_question_options_answer_summary_20260406-121506.csv
Baseline generation time (min): 0.0


...

steering generations:  98%|█████████▊| 19700/20000 [4:42:02<06:21,  1.27s/gen, dataset=implicit_full, layer=26, signed_strength=-32]

Checkpoint saved (implicit_full-layer26-steer_long_term-32): /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_vast/20260406-121506/mmraz_qwen3_probe_artifact_steering_question_options_answer_summary_20260406-121506.csv
[implicit_full] mm layer=26 strength=32 steer_long_term (computed_now): long=0.637 imm=0.363 fallback=0.047 logprob_margin=0.3479


steering generations: 100%|██████████| 20000/20000 [4:48:29<00:00,  1.16gen/s, dataset=implicit_full, layer=26, signed_strength=-32]

Checkpoint saved (implicit_full-layer26-steer_immediate-32): /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_vast/20260406-121506/mmraz_qwen3_probe_artifact_steering_question_options_answer_summary_20260406-121506.csv
[implicit_full] mm layer=26 strength=32 steer_immediate (computed_now): long=0.557 imm=0.443 fallback=0.093 logprob_margin=-0.0117
Steering generation time (min): 288.5


Checkpoint saved (final): /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_vast/20260406-121506/mmraz_qwen3_probe_artifact_steering_question_options_answer_summary_20260406-121506.csv
Saved summary    : /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_vast/20260406-121506/mmraz_qwen3_probe_artifact_steering_question_options_answer_summary_20260406-121506.csv
Saved logs       : /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_vast/20260406-121506/mmraz_qwen3_probe_artifact_steering_question_options_answer_logs_20260406-121506.csv
Saved probe slice: /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_vast/20260406-121506/mmraz_qwen3_probe_artifact_steering_question_options_answer_probe_slice_20260406-121506.csv
Saved reuse map  : /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_vast/20260406-121506/mmraz_qwen3_probe_artifact_steering_question_options_answer_

In [4]:
print('Output dir:', result['output_dir'])
display(result['probe_slice_df'])
display(result['baseline_summary_df'])
display(result['reuse_coverage_df'])
display(result['artifact_index_df'])


Output dir: /workspace/results/qwen3_4b_probe_artifact_steering_question_options_answer_vast/20260406-121506


,train_regime,feature_name,vector_key,layer,raw_norm,steering_norm
0,explicit_train_only,mean_answer_tokens,mm_steering_vectors,10,8.771654,1.0
1,explicit_train_only,mean_answer_tokens,mm_steering_vectors,14,10.132462,1.0
2,explicit_train_only,mean_answer_tokens,mm_steering_vectors,18,11.991506,1.0
3,explicit_train_only,mean_answer_tokens,mm_steering_vectors,22,20.724382,1.0
4,explicit_train_only,mean_answer_tokens,mm_steering_vectors,26,33.307693,1.0


,dataset,probe_variant,condition,layer,strength,signed_strength,raw_vector_norm,steering_vector_norm,baseline_prop_choose_long_term,baseline_prop_choose_immediate,...,n_no_fallback_prompts,prop_choose_long_term,prop_choose_immediate,prop_choose_long_term_no_fallback,prop_choose_immediate_no_fallback,fallback_rate,direct_parse_rate,mean_long_minus_immediate_avg_logprob,mean_long_minus_immediate_sum_logprob,prop_logprob_prefers_long_term
0,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,0.640000,0.360000,...,85,0.640000,0.360000,0.705882,0.294118,0.15,0.85,0.689644,0.381820,0.57
1,implicit_full,baseline,baseline,-1,0.0,0.0,NaN,NaN,0.586667,0.413333,...,276,0.586667,0.413333,0.605072,0.394928,0.08,0.92,0.197873,0.858146,0.60


,dataset,condition,layer,strength,signed_strength,n_prompts,reused,source_path
0,explicit_test,baseline,-1,0.0,0.0,100,True,/workspace/results/qwen3_4b_probe_artifact_ste...
1,explicit_test,steer_long_term,10,2.0,2.0,100,True,/workspace/results/qwen3_4b_probe_artifact_ste...
2,explicit_test,steer_immediate,10,2.0,-2.0,100,True,/workspace/results/qwen3_4b_probe_artifact_ste...
3,explicit_test,steer_long_term,10,4.0,4.0,100,True,/workspace/results/qwen3_4b_probe_artifact_ste...
4,explicit_test,steer_immediate,10,4.0,-4.0,100,True,/workspace/results/qwen3_4b_probe_artifact_ste...
...,...,...,...,...,...,...,...,...
97,implicit_full,steer_immediate,26,8.0,-8.0,300,False,NaN
98,implicit_full,steer_long_term,26,16.0,16.0,300,False,NaN
99,implicit_full,steer_immediate,26,16.0,-16.0,300,False,NaN
100,implicit_full,steer_long_term,26,32.0,32.0,300,False,NaN


,artifact,path
0,summary_csv,/workspace/results/qwen3_4b_probe_artifact_ste...
1,logs_csv,/workspace/results/qwen3_4b_probe_artifact_ste...
2,probe_slice_csv,/workspace/results/qwen3_4b_probe_artifact_ste...
3,reuse_coverage_csv,/workspace/results/qwen3_4b_probe_artifact_ste...
4,config_json,/workspace/results/qwen3_4b_probe_artifact_ste...
5,explicit_test_heatmap_png,/workspace/results/qwen3_4b_probe_artifact_ste...
6,implicit_full_heatmap_png,/workspace/results/qwen3_4b_probe_artifact_ste...


In [5]:
result['summary_df'].head(20)


,dataset,probe_variant,condition,layer,strength,signed_strength,raw_vector_norm,steering_vector_norm,baseline_prop_choose_long_term,baseline_prop_choose_immediate,...,n_no_fallback_prompts,prop_choose_long_term,prop_choose_immediate,prop_choose_long_term_no_fallback,prop_choose_immediate_no_fallback,fallback_rate,direct_parse_rate,mean_long_minus_immediate_avg_logprob,mean_long_minus_immediate_sum_logprob,prop_logprob_prefers_long_term
0,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,0.640000,0.360000,...,85,0.640000,0.360000,0.705882,0.294118,0.15,0.85,0.689644,0.381820,0.57
1,implicit_full,baseline,baseline,-1,0.0,0.0,NaN,NaN,0.586667,0.413333,...,276,0.586667,0.413333,0.605072,0.394928,0.08,0.92,0.197873,0.858146,0.60
2,explicit_test,mm,steer_long_term,10,2.0,2.0,8.771654,1.0,0.640000,0.360000,...,79,0.640000,0.360000,0.746835,0.253165,0.21,0.79,0.685718,0.378509,0.57
3,explicit_test,mm,steer_immediate,10,2.0,-2.0,8.771654,1.0,0.640000,0.360000,...,83,0.640000,0.360000,0.722892,0.277108,0.17,0.83,0.687882,0.368582,0.57
4,explicit_test,mm,steer_long_term,10,4.0,4.0,8.771654,1.0,0.640000,0.360000,...,80,0.650000,0.350000,0.762500,0.237500,0.20,0.80,0.678562,0.366785,0.57
5,explicit_test,mm,steer_immediate,10,4.0,-4.0,8.771654,1.0,0.640000,0.360000,...,81,0.630000,0.370000,0.728395,0.271605,0.19,0.81,0.678644,0.333406,0.57
6,explicit_test,mm,steer_long_term,10,8.0,8.0,8.771654,1.0,0.640000,0.360000,...,80,0.670000,0.330000,0.787500,0.212500,0.20,0.80,0.664561,0.349662,0.56
7,explicit_test,mm,steer_immediate,10,8.0,-8.0,8.771654,1.0,0.640000,0.360000,...,86,0.660000,0.340000,0.732558,0.267442,0.14,0.86,0.640643,0.204029,0.57
8,explicit_test,mm,steer_long_term,10,16.0,16.0,8.771654,1.0,0.640000,0.360000,...,77,0.710000,0.290000,0.831169,0.168831,0.23,0.77,0.658397,0.376816,0.56
9,explicit_test,mm,steer_immediate,10,16.0,-16.0,8.771654,1.0,0.640000,0.360000,...,89,0.590000,0.410000,0.640449,0.359551,0.11,0.89,0.552601,-0.068308,0.57


In [6]:
result['logs_df'].head(20)


,dataset,probe_variant,condition,layer,strength,signed_strength,raw_vector_norm,steering_vector_norm,prompt_idx,question,...,immediate_token_count,long_term_sum_logprob,long_term_avg_logprob,long_term_token_count,long_minus_immediate_sum_logprob,long_minus_immediate_avg_logprob,logprob_prefers_long_term,source_path,result_source,reused_from_path
0,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,0,"When developing a roadmap for this initiative,...",...,11,-38.766693,-4.307410,9,-12.421978,-1.912436,0.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
1,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,1,"When setting milestones, focus on:",...,7,-39.382812,-6.563802,6,0.335938,-0.889695,0.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
2,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,2,"When drafting the plan, consider:",...,7,-21.468710,-3.578118,6,8.308702,0.675798,1.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
3,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,3,Resource planning aligns with:,...,5,-31.069576,-6.213915,5,2.868950,0.573790,1.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
4,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,4,Contingency planning addresses:,...,8,-31.743282,-3.967910,8,0.028427,0.003553,1.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
5,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,5,The planning framework emphasizes:,...,7,-36.132820,-4.516603,8,7.093742,1.658621,1.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
6,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,6,Choose the option that delivers:,...,7,-37.531563,-5.361652,7,8.218750,1.174108,1.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
7,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,7,"When faced with uncertainty, choose:",...,6,-31.235968,-5.205995,6,6.468748,1.078125,1.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
8,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,8,The decision tree evaluates:,...,6,-40.843750,-8.168750,5,-0.078125,-1.374479,0.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
9,explicit_test,baseline,baseline,-1,0.0,0.0,NaN,NaN,9,Decision reversibility matters over:,...,4,-23.625557,-5.906389,4,18.500000,4.625000,1.0,/workspace/results/qwen3_4b_probe_artifact_ste...,reused_existing,/workspace/results/qwen3_4b_probe_artifact_ste...
